In [ ]:
import torch
import numpy as np
import neuropythy as ny
import matplotlib.pyplot as plt

import sys
sys.path.append('src')

# Import the visual_autolabel library from the src/ directory:
import visual_autolabel as val
# Also the volume data subpackage; we need to set the volume data cache path:
import visual_autolabel.voldata as voldata
voldata.dataset_cache_path = '/data/visual-autolabel/volumetric/data'

In [ ]:
# We make a 2D Image Cache object in order to make the transform matrices.
imcache = val.benson2025.hcp.HCPImageCache()

In [ ]:
# Test the code:
# We import the appropriate functions and try to create a transform matrix;
# then we use the transform matrix to convert a set of 3D subject data into
# 2D flatmap images.

from visual_autolabel.voldata import load_subject_affine, load_subject_data

# The subject and rater we'll use:
targ = {'subject': 100610, 'rater': 'A1'}

# Load volume data to project to a 2D image:
dat = load_subject_data(targ['subject'])

# Make the subject's affine matrix (used to make the transform).
aff = load_subject_affine(
    targ['subject'],
    zoom=0.5,
    subindex=(slice(2,-2), slice(8, 256+8), slice(2,-2)),
    forget=False)
# Make the sparse matrix transform:
S = imcache.volume_to_image_matrix(aff, dat[0].shape[1:], targ)

out = np.reshape((S @ np.reshape(dat[0], (3,-1)).T).T, (3,512,1024))

# Plot these 2D images to see if it worked.
(fig,axs) = plt.subplots(3,1, figsize=(6,4), dpi=288)
for (ax,im) in zip(axs, out):
    ax.imshow(im, cmap='gray')
    ax.axis('off')

plt.show()

In [ ]:
# Go through the subjects and save an affine matrix for each.

import importlib

dset = val.benson2025.hcp.HCPDataset(inputs=('curvature',), outputs=('V1',))

finished_sids = set()
for targ in dset.targets:
    sid = targ['subject']
    if sid in finished_sids:
        print(f' ✔ {sid}')
        continue
    else:
        print(f' - {sid}')
        finished_sids.add(sid)
    sub = ny.data['hcp_lines'].subjects[sid]
    aff = load_subject_affine(
        targ['subject'],
        zoom=0.5,
        subindex=(slice(2,-2), slice(8, 256+8), slice(2,-2)))
    S = dset.image_cache.volume_to_image_matrix(aff, (128,128,128), targ)
    torch.save(
        S,
        f"/data/visual-autolabel/volumetric/data/affines/{sid}.pt")
    # Make sure to clear cache by reloading the libraries:
    ny = ny.reload_neuropythy()
    val = importlib.reload(val)
    # We need to generate a new imcache after reloading visual_autolabel.
    dset = val.benson2025.hcp.HCPDataset(inputs=('curvature',), outputs=('V1',))